In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.lines import Line2D

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.964, 0.912, 0.849, 0.691, 0.314]
B_eff = 0.013
B_cs = 87
L1 = 25 #ab-1，1 ab⁻¹ = 1000 fb⁻¹
L2 = 150 #ab-1
# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths.csv')

# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP['xs_mumu_prompt'] = pho_ALP['xs_mumu_prompt'].astype('float32')
pho_ALP['Lambda'] = 1000 / pho_ALP['Lambda']

In [ ]:
background_14_os = 87 * 0.013 * 1000 * 1000

In [ ]:
values_os

In [ ]:
S_grid_14_os

In [ ]:
# OS 网格
ma_14_os = np.linspace(pho_ALP['ma'].min(), pho_ALP['ma'].max(), 10000)
Lambda_14_os = np.linspace(pho_ALP['Lambda'].min(), pho_ALP['Lambda'].max(), 350)
ma_grid_14_os, Lambda_grid_14_os = np.meshgrid(ma_14_os, Lambda_14_os)
efficiency_14_os = np.interp(ma_grid_14_os, ma, S_eff)
points_os = pho_ALP[['ma', 'Lambda']].values
values_os = pho_ALP['xs_mumu_prompt'].values
S_grid_14_os = griddata(points_os, values_os, (ma_grid_14_os, Lambda_grid_14_os), method='linear', fill_value=0) * 25 * efficiency_14_os * 1000 * 1000
mask = (ma_grid_14_os < 0.21)  # ma[2] = 50（根据你的 ma = [10, 30, 50, ...]）
S_grid_14_os[mask] = 0  # 强制置零
Z_grid_14_os = S_grid_14_os / np.sqrt(S_grid_14_os + background_14_os)


# 绘制图形
plt.figure(figsize=(10, 7))

# 绘制 OS 等高线及误差带
contour_os = plt.contour(ma_grid_14_os, Lambda_grid_14_os, Z_grid_14_os, levels=[2], colors=['r'])


# 设置轴标签和坐标范围
plt.xlabel(R'$m_{N}$ (TeV)', fontsize=30)
plt.ylabel(R'$s_{\alpha}$', fontsize=30)
plt.grid(True)
#plt.xlim(1.1, 6.0)  # GeV 范围转换成 TeV
plt.ylim(1e-4,1e0)
plt.yscale('log')
plt.tick_params(axis='both', which='major', labelsize=24)
plt.xscale('log')

# 手动添加图例
legend_elements = [
    Line2D([0], [0], color='r', lw=2, label=R'OS 1$\sigma$ C.L.'),
    Line2D([0], [0], color='r', lw=2, linestyle='--', label=R'OS 1$\sigma$ boundary'),
]
plt.legend(handles=legend_elements, loc='center right', fontsize=18)

# 添加其他文本
#plt.text(4.5, 0.2, R'$m_{W_{R}} = 6.5~$ TeV', fontsize=22, color='black')
#plt.text(4.5, 0.1, R'$\mathcal{L}$  = 3 ab$^{-1}$', fontsize=22, color='black')
#plt.text(4.5, 0.15, R'$\sqrt{S}$ = 14 TeV', fontsize=22, color='black')
#plt.text(4.5, 0.05, R'$ee$ final state', fontsize=22, color='black')
plt.tight_layout()

# 保存并显示图像
plt.savefig('./14TeV.png', dpi=300)
plt.show()
